# 07_joining

This notebook creates the joined contract master table used downstream for feature engineering, weak supervision, and gold-label evaluation.

Main tasks:
1. Load the cleaned source tables from the project data folders
2. Join supplier bridge, financials, ESG, news, stocks, and macro indexes
3. Create a simple manually maintained gold-label table
4. Join `gold_y` onto the final contract table
5. Save both:
   - joined contract table without gold labels
   - joined contract table with gold labels
   - manual gold label table

## Imports and project path setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/Thomas/Desktop/Master Thesis
src_path: /Users/Thomas/Desktop/Master Thesis/src


In [2]:
import pandas as pd
import numpy as np

from master_thesis.data_utils import load_raw, save_processed

## File names

Update if we change raw files naming

In [3]:
RAW_FILES = {
    "contract_year": "contract_year_final.csv",
    "supplier_bridge": "dim_supplier.csv",
    "financials": "financials_clean.csv",
    "esg": "esg_yearly.csv",
    "news": "news.csv",
    "stocks": "Stock_view.csv",      # update if your file is named stock_view.csv
    "indexes": "macro_indexs.csv",   # update if your file is named macro_indexes.csv
}

## Load core tables

In [4]:
df_contract_year = load_raw(RAW_FILES["contract_year"], low_memory=False)
df_suppliers_bridge = load_raw(RAW_FILES["supplier_bridge"], low_memory=False)
df_financials_clean = load_raw(RAW_FILES["financials"], low_memory=False)
df_esg_yearly = load_raw(RAW_FILES["esg"], low_memory=False)
df_news = load_raw(RAW_FILES["news"], low_memory=False)
df_stocks = load_raw(RAW_FILES["stocks"], low_memory=False)
df_indexes = load_raw(RAW_FILES["indexes"], low_memory=False)

print("contract_year:", df_contract_year.shape)
print("supplier_bridge:", df_suppliers_bridge.shape)
print("financials:", df_financials_clean.shape)
print("esg:", df_esg_yearly.shape)
print("news:", df_news.shape)
print("stocks:", df_stocks.shape)
print("indexes:", df_indexes.shape)

contract_year: (9201, 37)
supplier_bridge: (1000, 51)
financials: (3529, 54)
esg: (2093, 16)
news: (835, 11)
stocks: (30288, 27)
indexes: (1635, 12)


## Helper functions

In [5]:
def clean_id_like_column(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

def clean_country_name(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    mapping = {
        "RUSSIAN FEDERATION": "Russia",
        "UNITED STATES": "United States",
        "Novo Nordisk Saudi Arabia": "Saudi Arabia",
    }

    if value in mapping:
        return mapping[value]

    if value.isupper():
        return value.title()

    return value

def build_manual_gold_table(gold_contract_map: dict) -> pd.DataFrame:
    rows = []

    for department, labels in gold_contract_map.items():
        for contract_id in labels.get("yes", []):
            rows.append({
                "contract_id": str(contract_id),
                "gold_y": 1,
                "gold_department": department,
            })
        for contract_id in labels.get("no", []):
            rows.append({
                "contract_id": str(contract_id),
                "gold_y": 0,
                "gold_department": department,
            })

    df_gold = pd.DataFrame(rows).drop_duplicates(subset=["contract_id"])
    return df_gold

## Contract-year base table

In [6]:
df_contract = df_contract_year.copy()

df_contract["supplier_number"] = clean_id_like_column(df_contract["supplier_number"])
df_contract["contract_id"] = clean_id_like_column(df_contract["contract_id"])

df_contract["observation_year"] = pd.to_numeric(
    df_contract["observation_year"],
    errors="coerce",
).astype("Int64")

print("Shape:", df_contract.shape)
print("Unique contracts:", df_contract["contract_id"].nunique())
print("Duplicate contract-year rows:", df_contract.duplicated(["contract_id", "observation_year"]).sum())
display(df_contract.head())

Shape: (9201, 37)
Unique contracts: 2209
Duplicate contract-year rows: 0


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,years_to_expiry,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,False,2025,2018,2018,9,0,5y_plus,"Distribution, GxP services & Energy","Quality, Production Services & Supplies"
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,False,2025,2018,2019,8,1,5y_plus,"Distribution, GxP services & Energy","Quality, Production Services & Supplies"
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,False,2025,2018,2020,7,2,5y_plus,"Distribution, GxP services & Energy","Quality, Production Services & Supplies"
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,False,2025,2018,2021,6,3,5y_plus,"Distribution, GxP services & Energy","Quality, Production Services & Supplies"
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,False,2025,2018,2022,5,4,3_to_5y,"Distribution, GxP services & Energy","Quality, Production Services & Supplies"


## Supplier bridge join

In [7]:
df_bridge = df_suppliers_bridge.copy()

df_bridge["supplier_key"] = clean_id_like_column(df_bridge["supplier_key"])
df_bridge["moodys_bvd_id"] = df_bridge["moodys_bvd_id"].astype("string").str.strip()

df_bridge_slim = (
    df_bridge[["supplier_key", "moodys_bvd_id"]]
    .drop_duplicates()
    .copy()
)

print("Duplicate bridge supplier_key rows:", df_bridge_slim.duplicated(["supplier_key"]).sum())

Duplicate bridge supplier_key rows: 0


In [8]:
df_contract_bvd = (
    df_contract.merge(
        df_bridge_slim,
        left_on="supplier_number",
        right_on="supplier_key",
        how="left",
    )
    .drop(columns=["supplier_key"])
)

print("Rows:", len(df_contract_bvd))
print("Matched to BvD ID:", df_contract_bvd["moodys_bvd_id"].notna().sum())
print("Duplicate contract-year rows:", df_contract_bvd.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
Matched to BvD ID: 4634
Duplicate contract-year rows: 0


## Financials join

In [9]:
df_fin = df_financials_clean.copy()

df_fin["moodys_bvd_id"] = df_fin["moodys_bvd_id"].astype("string").str.strip()
df_fin["Join_Year"] = pd.to_numeric(df_fin["Join_Year"], errors="coerce").astype("Int64")

fin_key_cols = ["moodys_bvd_id", "Join_Year"]
fin_other_cols = [c for c in df_fin.columns if c not in fin_key_cols]

df_fin_prefixed = df_fin.rename(columns={c: f"fin_{c}" for c in fin_other_cols})

print("Duplicate financial company-year:", df_fin.duplicated(["moodys_bvd_id", "Join_Year"]).sum())

Duplicate financial company-year: 0


In [10]:
df_contract_fin = (
    df_contract_bvd.merge(
        df_fin_prefixed,
        left_on=["moodys_bvd_id", "observation_year"],
        right_on=["moodys_bvd_id", "Join_Year"],
        how="left",
    )
    .drop(columns=["Join_Year"])
)

print("Rows:", len(df_contract_fin))
print("Financial matches:", df_contract_fin["fin_Operating_revenue_Turnover_th_DKK"].notna().sum())
print("Duplicate contract-year rows:", df_contract_fin.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
Financial matches: 1281
Duplicate contract-year rows: 0


## ESG join

In [11]:
df_esg = df_esg_yearly.copy()

df_esg["moodys_bvd_id"] = df_esg["moodys_bvd_id"].astype("string").str.strip()
df_esg["Join_Year"] = pd.to_numeric(df_esg["Join_Year"], errors="coerce").astype("Int64")

esg_key_cols = ["moodys_bvd_id", "Join_Year"]
esg_other_cols = [c for c in df_esg.columns if c not in esg_key_cols]

df_esg_prefixed = df_esg.rename(columns={c: f"esg_{c}" for c in esg_other_cols})

df_contract_fin_esg = (
    df_contract_fin.merge(
        df_esg_prefixed,
        left_on=["moodys_bvd_id", "observation_year"],
        right_on=["moodys_bvd_id", "Join_Year"],
        how="left",
    )
    .drop(columns=["Join_Year"])
)

print("Rows:", len(df_contract_fin_esg))
print("ESG matches:", df_contract_fin_esg["esg_esg_overall"].notna().sum())
print("Duplicate contract-year rows:", df_contract_fin_esg.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
ESG matches: 1012
Duplicate contract-year rows: 0


## News join

In [12]:
df_news_view = df_news.copy()

df_news_view["supplier_number"] = clean_id_like_column(df_news_view["supplier_number"])
df_news_view["Join_Year"] = pd.to_numeric(df_news_view["Join_Year"], errors="coerce").astype("Int64")

news_key_cols = ["supplier_number", "Join_Year"]
news_other_cols = [c for c in df_news_view.columns if c not in news_key_cols]

df_news_prefixed = df_news_view.rename(columns={c: f"news_{c}" for c in news_other_cols})

df_feature_matrix = (
    df_contract_fin_esg.merge(
        df_news_prefixed,
        left_on=["supplier_number", "observation_year"],
        right_on=["supplier_number", "Join_Year"],
        how="left",
    )
    .drop(columns=["Join_Year"])
)

print("Rows:", len(df_feature_matrix))
print("News matches:", df_feature_matrix["news_article_count"].notna().sum())
print("Duplicate contract-year rows:", df_feature_matrix.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
News matches: 1185
Duplicate contract-year rows: 0


## Stocks join

In [13]:
df_stocks_clean = df_stocks.rename(
    columns={
        "BvD ID number": "moodys_bvd_id",
        "Join_Year": "observation_year",
    }
).copy()

df_stocks_clean = df_stocks_clean.drop(columns=["Unnamed: 0"], errors="ignore")

if "moodys_bvd_id" in df_stocks_clean.columns:
    df_stocks_clean["moodys_bvd_id"] = df_stocks_clean["moodys_bvd_id"].astype("string").str.strip()

if "observation_year" in df_stocks_clean.columns:
    df_stocks_clean["observation_year"] = pd.to_numeric(df_stocks_clean["observation_year"], errors="coerce").astype("Int64")

join_keys = ["moodys_bvd_id", "observation_year"]

stock_new_columns = [
    col for col in df_stocks_clean.columns
    if col not in df_feature_matrix.columns or col in join_keys
]

df_stocks_to_merge = df_stocks_clean[stock_new_columns].copy()

print("Duplicate stock rows on join keys:", df_stocks_to_merge.duplicated(subset=join_keys).sum())

Duplicate stock rows on join keys: 0


In [14]:
df_feature_matrix_enriched = df_feature_matrix.merge(
    df_stocks_to_merge,
    on=join_keys,
    how="left",
)

print("Original shape:", df_feature_matrix.shape)
print("Enriched shape:", df_feature_matrix_enriched.shape)

added_stock_columns = [c for c in df_feature_matrix_enriched.columns if c not in df_feature_matrix.columns]
print("Added stock columns:")
print(added_stock_columns)

Original shape: (9201, 113)
Enriched shape: (9201, 137)
Added stock columns:
['Company name Latin alphabet', 'Year', 'avg_vol', 'std_vol', 'max_vol', 'min_vol', 'vol_stability_score', 'vol_shock_ratio', 'vol_trend_slope', 'Risk level', 'avg_market_cap', 'market_cap_volatility', 'supplier_sector', 'moodys_risk_rating', 'Price_trends_52 weeks_%', 'market_beta_1y', 'Earnings_per_share_DKK', 'Book_value_per_share_DKK', 'Shares outstanding', 'Current_market_capitalisation_DKK', 'avg_closing_price', 'price_volatility_score', 'price_trend_slope', 'Risk level_stock_closing_price']


## Macro index join

In [15]:
df_left = df_feature_matrix_enriched.copy()
df_right = df_indexes.copy()

df_right = df_right.drop(columns=["Unnamed: 0"], errors="ignore")
df_right = df_right.rename(columns={"Year": "observation_year"})

df_left["company_country_clean"] = df_left["company_country"].apply(clean_country_name)
df_right["Country_clean"] = df_right["Country"].astype(str).str.strip()
df_right = df_right.rename(columns={"Country_clean": "company_country_clean"})

df_right["observation_year"] = pd.to_numeric(df_right["observation_year"], errors="coerce").astype("Int64")

join_keys = ["company_country_clean", "observation_year"]

macro_cols_to_merge = [
    col for col in df_right.columns
    if col not in df_left.columns or col in join_keys
]

df_right_to_merge = df_right[macro_cols_to_merge].copy()

print("Duplicate macro rows on join keys:", df_right_to_merge.duplicated(subset=join_keys).sum())

Duplicate macro rows on join keys: 0


In [16]:
df_contract_final_no_spend = df_left.merge(
    df_right_to_merge,
    on=join_keys,
    how="left",
)

added_macro_columns = [c for c in df_contract_final_no_spend.columns if c not in df_feature_matrix_enriched.columns]

print("Added macro columns:")
print(added_macro_columns)

print("Final shape:", df_contract_final_no_spend.shape)
print("Rows missing macro match (using LPI_Score):", df_contract_final_no_spend["LPI_Score"].isna().sum())
print("Macro match rate:", f"{(1 - df_contract_final_no_spend['LPI_Score'].isna().mean()):.2%}")

display(df_contract_final_no_spend.head())

Added macro columns:
['company_country_clean', 'Country', 'Code', 'LPI_Score', 'Customs', 'Infrastructure', 'International_Shipments', 'Logistics_Competence', 'Tracking_Tracing', 'Timeliness', 'PPI_Value']
Final shape: (9201, 148)
Rows missing macro match (using LPI_Score): 67
Macro match rate: 99.27%


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,PPI_Value
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,DNK,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,84.4
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,83.9
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,79.9
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,100.0
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,147.0


## Manual gold-label table

This is meant to be easy to maintain.

Add or remove contract ids directly inside the dictionary below.  
Each contract id should appear only once overall.

In [17]:
gold_contract_map = {
    "Bioprocessing & Excipients - Cell Culture Media": {
        "yes": [504149, 612555],
        "no": [346312, 1877],
    },
    "Logistics": {
        "yes": [192325, 541269],
        "no": [536319, 598142],
    },
    "Global Strategic Outsourcing & Devices I Devices & Needles": {
        "yes": [4822, 557, 565],
        "no": [483178, 25777],
    },
}

df_gold_manual = build_manual_gold_table(gold_contract_map)

print("Gold labels shape:", df_gold_manual.shape)
display(df_gold_manual.sort_values(["gold_department", "gold_y", "contract_id"]))

Gold labels shape: (13, 3)


,contract_id,gold_y,gold_department
3,1877,0,Bioprocessing & Excipients - Cell Culture Media
2,346312,0,Bioprocessing & Excipients - Cell Culture Media
0,504149,1,Bioprocessing & Excipients - Cell Culture Media
1,612555,1,Bioprocessing & Excipients - Cell Culture Media
12,25777,0,Global Strategic Outsourcing & Devices I Devic...
11,483178,0,Global Strategic Outsourcing & Devices I Devic...
8,4822,1,Global Strategic Outsourcing & Devices I Devic...
9,557,1,Global Strategic Outsourcing & Devices I Devic...
10,565,1,Global Strategic Outsourcing & Devices I Devic...
6,536319,0,Logistics


## Join gold labels onto final contract table

In [18]:
df_contract_final_no_spend["contract_id"] = clean_id_like_column(df_contract_final_no_spend["contract_id"])
df_gold_manual["contract_id"] = clean_id_like_column(df_gold_manual["contract_id"])

df_contract_with_gold = df_contract_final_no_spend.merge(
    df_gold_manual,
    on="contract_id",
    how="left",
)

print("Joined shape:", df_contract_with_gold.shape)
print("Rows with gold_y:", df_contract_with_gold["gold_y"].notna().sum())
print("Unique gold-labeled contracts in final table:", df_contract_with_gold.loc[df_contract_with_gold["gold_y"].notna(), "contract_id"].nunique())
display(df_contract_with_gold[["contract_id", "department", "gold_y", "gold_department"]].dropna(subset=["gold_y"]).drop_duplicates().sort_values(["gold_department", "gold_y", "contract_id"]))

Joined shape: (9201, 150)
Rows with gold_y: 22
Unique gold-labeled contracts in final table: 2


,contract_id,department,gold_y,gold_department
3080,557,Devices & Needles,1.0,Global Strategic Outsourcing & Devices I Devic...
3157,565,Devices & Needles,1.0,Global Strategic Outsourcing & Devices I Devic...


## Check unmatched manual gold contract ids

These are contract ids present in your manual gold table but not found in the joined contract table.

In [19]:
gold_ids = set(df_gold_manual["contract_id"].unique())
final_ids = set(df_contract_final_no_spend["contract_id"].unique())

unmatched_gold_ids = sorted(gold_ids - final_ids)

print("Unmatched gold contract ids:", unmatched_gold_ids)

Unmatched gold contract ids: ['1877', '192325', '25777', '346312', '4822', '483178', '504149', '536319', '541269', '598142', '612555']


## Save outputs

Files saved:
- `contract_final_no_spend.csv`
- `gold_labels_manual.csv`
- `contract_with_gold.csv`

In [20]:
save_processed(df_contract_final_no_spend, "contract_final_no_spend.csv")
save_processed(df_gold_manual, "gold_labels_manual.csv")
save_processed(df_contract_with_gold, "contract_with_gold.csv")

print("Saved:")
print("- Data/processed/contract_final_no_spend.csv")
print("- Data/processed/gold_labels_manual.csv")
print("- Data/processed/contract_with_gold.csv")

Saved:
- Data/processed/contract_final_no_spend.csv
- Data/processed/gold_labels_manual.csv
- Data/processed/contract_with_gold.csv


In [21]:
gold_contracts = [
    504149, 612555, 346312, 1877,
    192325, 541269, 536319, 598142,
    4822, 557, 565, 483178, 25777
]

In [22]:
gold_contracts_str = [str(x) for x in gold_contracts]

df_check = df_contract_final_no_spend.copy()

# Clean both columns
df_check["contract_id_clean"] = (
    df_check["contract_id"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

df_check["contract_number_clean"] = (
    df_check["contract_number"].astype(str)
    .str.replace("#", "", regex=False)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

# Check matches
df_matches = df_check[
    df_check["contract_id_clean"].isin(gold_contracts_str) |
    df_check["contract_number_clean"].isin(gold_contracts_str)
]

print("Total matches found:", len(df_matches))
print("Unique contracts matched:", df_matches["contract_id"].nunique())

df_matches[
    ["contract_id", "contract_number", "contract_name", "department"]
].drop_duplicates().sort_values("contract_number")

Total matches found: 45
Unique contracts matched: 7


,contract_id,contract_number,contract_name,department
3080,557,557,Elos Medtech Tianjin Co. Ltd_Master_20091117_OA,Devices & Needles
3157,565,565,Mesa Parts GmbH & Co. KG_Master_20051123_SA,Devices & Needles
3459,4821,4822,Flextronics_Master_20130409_OA,Devices & Needles
425,192312,192325,SkyNRG_Stand-Alone_2021_Partnership Agreement ...,Logistics
2932,483149,483178,RKT_Rodinger_Kunststoff_Technik_GmbH_Master_20...,Devices & Needles
5100,504361,504149,Stena Recycling A/S_Commercial Lease Agreement...,Bioprocessing and Excipients
194,541883,541269,KUEHNE + NAGEL A/S_MASTER_2017_NNAS_Primary_LS...,Logistics


In [23]:
df_check["match_on_id"] = df_check["contract_id_clean"].isin(gold_contracts_str)
df_check["match_on_number"] = df_check["contract_number_clean"].isin(gold_contracts_str)

df_debug = df_check[df_check["match_on_id"] | df_check["match_on_number"]]

df_debug[
    ["contract_id", "contract_number", "match_on_id", "match_on_number"]
].drop_duplicates()

,contract_id,contract_number,match_on_id,match_on_number
194,541883,541269,False,True
425,192312,192325,False,True
2932,483149,483178,False,True
3080,557,557,True,True
3157,565,565,True,True
3459,4821,4822,False,True
5100,504361,504149,False,True


In [24]:
found_ids = set(df_check.loc[df_check["match_on_id"], "contract_id_clean"])
found_numbers = set(df_check.loc[df_check["match_on_number"], "contract_number_clean"])

all_found = found_ids.union(found_numbers)

missing = sorted(set(gold_contracts_str) - all_found)

print("Missing contracts:", missing)

Missing contracts: ['1877', '25777', '346312', '536319', '598142', '612555']
